# Cross-resonance calibration workflow

This notebook sketches a practical CR bring-up for one coupled qubit pair:
prepare the single-qubit pulse set, obtain CR parameters, calibrate a
`ZX90` gate, inspect the resulting schedule, and validate it with Bell
measurements and interleaved RB.

## 1. Create an `Experiment`

Pick a known coupled pair for your device. The example below uses the
first two qubits from the selected subset, but you should replace them
with a pair that already has a calibrated CR path in your system.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [ ]:
if len(exp.qubit_labels) < 2:
    raise ValueError("Select at least two coupled qubits for CR calibration.")

cr_control, cr_target = exp.qubit_labels[:2]
cr_label = f"{cr_control}-{cr_target}"

print("control qubit:", cr_control)
print("target qubit:", cr_target)
print("pair label:", cr_label)

## 2. Connect to the setup

Connect before you prepare the single-qubit prerequisites or run any CR measurements.

In [ ]:
exp.connect()

## 3. Prepare the single-qubit prerequisites

Measure the baseline Rabi response, calibrate the single-qubit pulses, and build the readout classifier before you move to CR calibration.

In [ ]:
result_rabi = exp.obtain_rabi_params()
result_hpi = exp.calibrate_hpi_pulse()
result_pi = exp.calibrate_pi_pulse()
result_drag_hpi = exp.calibrate_drag_hpi_pulse()
result_drag_pi = exp.calibrate_drag_pi_pulse()
result_classifier = exp.build_classifier()

## 4. Obtain the CR interaction parameters

Use the CR parameter scan to estimate a usable operating point for the entangling drive.

In [ ]:
result_cr = exp.obtain_cr_params(cr_control, cr_target)

## 5. Calibrate the `ZX90` gate

Run the `ZX90` calibration once the CR interaction parameters are in place.

In [ ]:
result_cr_calib = exp.calibrate_zx90(
    control_qubit=cr_control,
    target_qubit=cr_target,
)

## 6. Inspect the cached `ZX90` schedule

Plot the calibrated schedule so you can inspect the pulse structure directly.

In [ ]:
zx90 = exp.zx90(cr_control, cr_target)

zx90.plot()

## 7. Repeat the calibrated `ZX90` sequence

Use a repeated-sequence check to confirm that the calibrated gate behaves sensibly.

In [ ]:
repeat_result = exp.repeat_sequence(zx90)

## 8. Run pulse tomography with the control qubit in `|0>`

Reconstruct the repeated `ZX90` operation with the control qubit initialized in `|0>`.

In [ ]:
pulse_tomography_0 = exp.pulse_tomography(
    zx90.repeated(4), initial_state={cr_control: "0"}
)

## 9. Run pulse tomography with the control qubit in `|1>`

Repeat the same tomography measurement with the control qubit initialized in `|1>`.

In [ ]:
pulse_tomography_1 = exp.pulse_tomography(
    zx90.repeated(4), initial_state={cr_control: "1"}
)

## 10. Measure the Bell state

Use a Bell-state measurement as a direct entangling-gate validation step.

In [ ]:
result_bell = exp.measure_bell_state(cr_control, cr_target)

## 11. Run interleaved randomized benchmarking

Estimate the `ZX90` gate error with interleaved RB.

In [ ]:
result_irb = exp.interleaved_randomized_benchmarking(
    cr_label,
    interleaved_clifford="ZX90",
    n_cliffords_range=np.arange(0, 21, 2),
    n_trials=30,
)

## 12. Save the calibration note

Save the final CR calibration once the validation steps look acceptable.

In [ ]:
exp.calib_note.save()